# Agent timing debug notebook

Reproduces every step of `Agent.answer()` inline with per-step wall-clock timing.  
The goal is to identify which stage consumes the `BATCH_TIMEOUT` budget.

## 1 — Imports & helpers (copied from agent_llm.py)

In [ ]:
import json, re, time, random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

_NUMBER_RE = re.compile(r"-?\d+(?:[.,]\d+)*")

def _parse_final_number(text):
    if "####" not in text:
        return float("nan")
    tail = text.rsplit("####", 1)[-1].strip()
    matches = _NUMBER_RE.findall(tail)
    if not matches:
        return float("nan")
    try:
        return float(matches[-1].replace(",", ""))
    except ValueError:
        return float("nan")

## 2 — Constants

Set `BATCH_TIMEOUT` to the value the competition platform enforces.

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"

SYSTEM_PROMPT = (
    "You are a careful math tutor. Solve the problem step by step, "
    "then write the final numeric answer on a new line prefixed by '####'."
)

MAX_NEW_TOKENS = 512

# ← set this to the real platform timeout (seconds per batch)
BATCH_TIMEOUT = 60

## 3 — Load model & tokenizer (timed)

This happens once in `Agent.__init__()` — outside the timeout window — but useful to confirm device placement.

In [ ]:
t0 = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, clean_up_tokenization_spaces=False)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

load_time = time.perf_counter() - t0
print(f"Model load time : {load_time:.1f}s")
print(f"Device          : {model.device}")
print(f"dtype           : {next(model.parameters()).dtype}")

## 4 — Sample a realistic batch from question_set.json

Mirrors exactly what `Env` sends: `EASY_PER_SUBSET` easy/medium + `HARD_PER_SUBSET` hard questions, same seed.

In [ ]:
EASY_PER_SUBSET = 1
HARD_PER_SUBSET = 1
SEED = 0

with open("question_set.json") as f:
    items = json.load(f)

easy = [it for it in items if it.get("difficulty", "easy") != "hard"]
hard = [it for it in items if it.get("difficulty") == "hard"]

rng = random.Random(SEED)
rng.shuffle(easy)
rng.shuffle(hard)

batch_items = easy[:EASY_PER_SUBSET] + hard[:HARD_PER_SUBSET]
rng.shuffle(batch_items)

questions = [it["question"] for it in batch_items]
gold_answers = [it["answer"] for it in batch_items]

print(f"Batch size: {len(questions)}")
for i, (q, a) in enumerate(zip(questions, gold_answers)):
    label = batch_items[i].get("difficulty", "?")
    print(f"\n[{i}] ({label}) {q}")
    print(f"    gold: {a.split(chr(10))[-1]}")

## 5 — Step-by-step timing of answer()

Each sub-step is timed independently so you can pinpoint the bottleneck.

In [ ]:
# --- Step A: apply_chat_template ---
t0 = time.perf_counter()

prompts = []
for q in questions:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": q},
    ]
    prompts.append(tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    ))

template_time = time.perf_counter() - t0
print(f"[A] chat_template : {template_time*1000:.1f} ms")

# --- Step B: tokenise ---
t0 = time.perf_counter()

inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
).to(model.device)

tokenise_time = time.perf_counter() - t0
prompt_len = inputs["input_ids"].shape[1]
print(f"[B] tokenise      : {tokenise_time*1000:.1f} ms  |  padded prompt len = {prompt_len} tokens")
for i in range(len(questions)):
    real = int(inputs["attention_mask"][i].sum())
    print(f"     question [{i}]: {real} real tokens  ({prompt_len - real} padding)")

# --- Step C: model.generate ---
# No warm-up here — Agent.__init__() now does it before the timeout window.
t0 = time.perf_counter()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

if torch.cuda.is_available():
    torch.cuda.synchronize()

generate_time = time.perf_counter() - t0
tokens_generated = output_ids.shape[1] - prompt_len
tps = tokens_generated / generate_time
print(f"[C] generate      : {generate_time:.2f}s  |  {tokens_generated} new tokens  |  {tps:.1f} tok/s")

# --- Step D: decode ---
t0 = time.perf_counter()

replies = []
raw_outputs = []
for i in range(output_ids.shape[0]):
    text = tokenizer.decode(output_ids[i, prompt_len:], skip_special_tokens=True)
    raw_outputs.append(text)
    replies.append(_parse_final_number(text))

decode_time = time.perf_counter() - t0
print(f"[D] decode        : {decode_time*1000:.1f} ms")

## 6 — Generated outputs & parsed answers

In [ ]:
for i, (text, ans, gold) in enumerate(zip(raw_outputs, replies, gold_answers)):
    gold_val = _parse_final_number(gold)
    correct = abs(ans - gold_val) < 1e-6 if (ans == ans and gold_val is not None) else False
    print(f"=== Question [{i}] ===")
    print(text)
    print(f"=> parsed: {ans}   gold: {gold_val}   {'✓ correct' if correct else '✗ wrong'}")
    print()

## 7 — Timing summary vs timeout budget

In [ ]:
steps = [
    ("A  chat_template", template_time),
    ("B  tokenise",       tokenise_time),
    ("C  generate",       generate_time),
    ("D  decode",         decode_time),
]
total = sum(t for _, t in steps)

print(f"{'Step':<22} {'Time':>8}   {'%':>6}")
print("-" * 42)
for name, t in steps:
    print(f"{name:<22} {t:>7.3f}s  {100*t/total:>5.1f}%")
print("-" * 42)
print(f"{'TOTAL':<22} {total:>7.3f}s")
print()
print(f"BATCH_TIMEOUT budget : {BATCH_TIMEOUT}s")
margin = BATCH_TIMEOUT - total
if margin < 0:
    print(f"OVER budget by       : {-margin:.2f}s  ← WILL TIMEOUT")
else:
    print(f"Margin remaining     : {margin:.2f}s")

print()
overhead = template_time + tokenise_time + decode_time
budget_for_gen = BATCH_TIMEOUT - overhead
safe_tokens = int(budget_for_gen * tps)
print(f"At {tps:.0f} tok/s, max safe max_new_tokens = {safe_tokens}  (current = {MAX_NEW_TOKENS})")
if safe_tokens < MAX_NEW_TOKENS:
    print(f"  → reduce MAX_NEW_TOKENS to ~{safe_tokens} to stay within timeout")